In [ ]:
!pip install -q FlagEmbedding

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.8 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from tqdm.notebook import tqdm
from FlagEmbedding import BGEM3FlagModel

In [ ]:
model = BGEM3FlagModel('BAAI/bge-m3',
                       use_fp16=False,
                       devices="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
def cluster(method, eng_path):
    method_path = os.path.join(eng_path, method)
    if method != "greedy":
        data1 = pd.read_csv(os.path.join(method_path, f"english-corpus1.tsv.gz"), sep="\t", compression="gzip")
        data2 = pd.read_csv(os.path.join(method_path, f"english-corpus2.tsv.gz"), sep="\t", compression="gzip")
    else:
        data1 = pd.read_csv(os.path.join(method_path, f"english-corpus1.tsv.gz"), sep="\t", compression="gzip", header=None, names=('0', '1', '2'))
        data2 = pd.read_csv(os.path.join(method_path, f"english-corpus2.tsv.gz"), sep="\t", compression="gzip", header=None, names=('0', '1', '2'))
    print(data1.head())
    data1["corpus"] = [1 for _ in range(data1.shape[0])]

    data2["corpus"] = [2 for _ in range(data2.shape[0])]
    data = pd.concat((data1, data2))
    to_concat = pd.DataFrame()
    for target, target_df in tqdm(data.groupby("0")):
        definitions = target_df["2"].unique()

        if len(definitions) > 1:
            # print(len(definitions))
            clustering = AgglomerativeClustering(
                n_clusters=None, metric="cosine", linkage="complete", distance_threshold=0.2, compute_full_tree=True,
            )
            embeddings = model.encode(definitions.tolist(),
                                    max_length=2048, # If you don't need such a long length, you can set a smaller value to speed up the encoding process.
                                    )['dense_vecs']
            labels = clustering.fit_predict(embeddings)
            labels_def = []
            for label in np.unique(labels):
                idx = np.where(labels == label)[0]
                first_definition = Counter(definitions[idx]).most_common()[0][0]
                labels_def.append(first_definition)
            # print(labels_def)

            labels_list = []
            for i, row in target_df.iterrows():
                idx = np.where(definitions == row["2"])
                label = labels[idx].tolist()[0]
                labels_list.append(labels_def[label])
            target_df["labels"] = labels_list
        else:
            target_df["labels"] = target_df["2"]

        to_concat = pd.concat([to_concat, target_df])
    assert len(to_concat) == len(data)
    to_concat = to_concat.reset_index(drop=True)
    print(len(to_concat["2"].unique()))
    print(len(to_concat["labels"].unique()))
    data1 = to_concat[to_concat["corpus"]==1]
    data2 = to_concat[to_concat["corpus"]==2]
    return data1, data2

In [ ]:
eng_path = "drive/MyDrive/generated_definitions/english"

In [ ]:
dpo1, dpo2 = cluster("dpo", eng_path)

  0%|          | 0/37 [00:00<?, ?it/s]



pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 38.25it/s]


Inference Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Inference Embeddings:  50%|█████     | 1/2 [00:02<00:02,  2.33s/it]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.54s/it]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 82.89it/s]


Inference Embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

Inference Embeddings:  25%|██▌       | 1/4 [00:06<00:18,  6.26s/it]

Inference Embeddings:  50%|█████     | 2/4 [00:07<00:06,  3.15s/it]

Inference Embeddings:  75%|███████▌  | 3/4 [00:08<00:02,  2.07s/it]

Inference Embeddings: 100%|██████████| 4/4 [00:08<00:00,  2.10s/it]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 59.79it/s]


Inference Embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

Inference Embeddings:  25%|██▌       | 1/4 [00:07<00:21,  7.16s/it]

Inference Embeddings:  50%|█████     | 2/4 [00:08<00:07,  3.53s/it]

Inference Embeddings: 100%|██████████| 4/4 [00:08<00:00,  2.20s/it]


pre t

18869
5777


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
len(dpo1["2"].unique())

9346

In [ ]:
len(dpo1["labels"].unique())

3273

In [ ]:
dpo1["2"] = dpo1["labels"]
dpo1 = dpo1[["0", "1", "2"]]
dpo1.head()

,0,1,2
0,attack,"tiring breeze , and the grief - like and hollo...","An attempt to harm, damage, debilitate, or des..."
1,attack,"Ik saai , Ik hab mien dout , "" and having thus...","A violent assault, whether physical or verbal."
2,attack,"Having determined instantly , in his own mind ...","A violent engagement of bodies, either by land..."
3,attack,What is remarkable in these and in all the oth...,"A violent assault, for example on a building; ..."
4,attack,Just as they had taken their seats around the ...,An attempt to detract from the worth or credib...


In [ ]:
dpo1.to_csv("drive/MyDrive/generated_definitions/english/dpo-clustered/english-corpus1.tsv.gz", sep='\t', index=False, compression="gzip")


In [ ]:
dpo2["2"] = dpo2["labels"]
dpo2 = dpo2[["0", "1", "2"]]
dpo2.to_csv("drive/MyDrive/generated_definitions/english/dpo-clustered/english-corpus2.tsv.gz", sep='\t', index=False, compression="gzip")
dpo2.head()

,0,1,2
447,attack,"to listen to words of peace , and avert by ful...",An attempt to detract from the worth or credib...
448,attack,"The tabloid TV shows , like "" Hard Copy "" and ...","An attempt to cause damage, injury to, or deat..."
449,attack,We believe that the United States should pursu...,An attempt to detract from the worth or credib...
450,attack,"However , when one discovers that Snyder 's ga...",An attempt to detract from the worth or credib...
451,attack,Yet the reality today is worse than fiction : ...,"(Medicine) An acute myocardial infarction, som..."


In [ ]:
from collections import Counter

In [ ]:
def cluster_sep(method, eng_path, n):
    method_path = os.path.join(eng_path, method)
    if method != "greedy":
        data = pd.read_csv(os.path.join(method_path, f"english-corpus{n}.tsv.gz"), sep="\t", compression="gzip")
    else:
        data = pd.read_csv(os.path.join(method_path, f"english-corpus{n}.tsv.gz"), sep="\t", compression="gzip", header=None, names=('0', '1', '2'))
    to_concat = pd.DataFrame()
    for target, target_df in tqdm(data.groupby("0")):
        definitions = target_df["2"].unique()

        if len(definitions) > 1:
            # print(len(definitions))
            clustering = AgglomerativeClustering(
                n_clusters=None, metric="cosine", linkage="complete", distance_threshold=0.2, compute_full_tree=True,
            )
            embeddings = model.encode(definitions.tolist(),
                                    max_length=2048, # If you don't need such a long length, you can set a smaller value to speed up the encoding process.
                                    )['dense_vecs']
            labels = clustering.fit_predict(embeddings)
            labels_def = []
            for label in np.unique(labels):
                idx = np.where(labels == label)[0]
                first_definition = Counter(definitions[idx]).most_common()[0][0]
                labels_def.append(first_definition)
            # print(labels_def)

            labels_list = []
            for i, row in target_df.iterrows():
                idx = np.where(definitions == row["2"])
                label = labels[idx].tolist()[0]
                labels_list.append(labels_def[label])
            target_df["labels"] = labels_list
        else:
            target_df["labels"] = target_df["2"]

        to_concat = pd.concat([to_concat, target_df])
    assert len(to_concat) == len(data)
    to_concat = to_concat.reset_index(drop=True)
    print(len(to_concat["2"].unique()))
    print(len(to_concat["labels"].unique()))
    return to_concat

In [ ]:
for method in ("dpo", "periti_2024", "periti_2025"):
    print(method)
    new_dir = os.path.join(eng_path, f"{method}-clustered-sep/")
    os.makedirs(new_dir, exist_ok=True)
    for n in (1, 2):
        print(n)
        method_result = cluster_sep(method, eng_path, n)
        method_result["2"] = method_result["labels"]
        method_result.drop("labels", inplace=True, axis=1)
        method_result.to_csv(f"{new_dir}english-corpus{n}.tsv.gz", sep='\t', index=False, compression="gzip")
        method_result.head()

dpo
1


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 132.00it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:09<00:00,  4.94s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 114.26it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:07<00:00,  1.78s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 97.15it/s]

Inference Embeddings: 100%|██████████| 6/6 [00:11<00:00,  1.90s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 88.55it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:09<00:00,  1.89s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 97.89it/s]

Inference Embeddings: 100%|██████████| 6/6 [00:07<00:00,  1.30s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 67.21it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:08<00:00,  2.76s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 87.91it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:08<00:00,  1.61s/it]


9346
2836
2


  0%|          | 0/37 [00:00<?, ?it/s]

...


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 104.93it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:08<00:00,  4.17s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 71.92it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:08<00:00,  2.21s/it]

pre tokenize: 100%|██████████| 7/7 [00:00<00:00, 87.54it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:11<00:00,  1.60s/it]

pre tokenize: 100%|██████████| 7/7 [00:00<00:00, 75.25it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:14<00:00,  2.01s/it]

pre tokenize: 100%|██████████| 8/8 [00:00<00:00, 82.17it/s]

Inference Embeddings: 100%|██████████| 8/8 [00:12<00:00,  1.54s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 55.90it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:12<00:00,  3.08s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 64.79it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:10<00:00,  2.09s/it]


11336
3244
2


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 80.81it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:06<00:00,  3.48s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 88.16it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:09<00:00,  2.34s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 46.26it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:13<00:00,  4.55s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 57.56it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.69s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 91.61it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:06<00:00,  3.24s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 90.25it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:08<00:00,  1.78s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 67.63it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:09<00:00,  4.61s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 39.13it/s]

Inference Embeddings: 100%|█

12988
4389


In [ ]:
for method in ("greedy", "periti_2024", "periti_2025"):
    print(method)
    new_dir = os.path.join(eng_path, f"{method}-clustered/")
    os.makedirs(new_dir, exist_ok=True)

    data1, data2 = cluster(method, eng_path)
    data1["2"] = data1["labels"]
    data1 = data1[["0", "1", "2"]]
    data1.to_csv(f"{new_dir}english-corpus1.tsv.gz", sep='\t', index=False, compression="gzip")

    data2["2"] = data2["labels"]
    data2 = data2[["0", "1", "2"]]
    data2.to_csv(f"{new_dir}english-corpus2.tsv.gz", sep='\t', index=False, compression="gzip")


greedy
      0                                                  1  \
0  part  Most of the asp * ing men of that part of Alab...   
1  face  The faces are umajured , except that the color...   
2   tip  The moon was in its western field , white and ...   
3  tree  The well known fact that we may have a hundred...   
4  face  Rising upon the instant in his stirrups , the ...   

                                                   2  
0              A portion ; some portion ; a section.  
1                   The front surface of a painting.  
2                  To cover the top of ; to overlay.  
3  A plant that bears a given fruit or seed ; spe...  
4  The front of the skull, especially the front o...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 107.02it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:04<00:00,  1.16s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 95.81it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.35s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 23.14it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.37s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 131.07it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:04<00:00,  1.45s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 91.82it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.13s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 96.48it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.29s/it]

pre tokenize: 100%|██████████| 9/9 [00:00<00:00, 45.69it/s]

Inference Embeddings: 100%|██████████| 9/9 [00:10<00:00,  1.13s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 45.44it/s]

Inference Embeddings: 100%

33065
7119
periti_2024
      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  part  immaturity , and want of experience -- besides...   
4  word  immaturity , and want of experience -- besides...   

                                                   2  
0            A superior or governing body or member.  
1  The top of the head of a person, or the upper ...  
2  A unit of language that native speakers can id...  
3                             A fraction of a whole.  
4  A single distinct conceptual unit of language,...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 73.00it/s]

Inference Embeddings: 100%|██████████| 12/12 [01:23<00:00,  6.93s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 71.91it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:10<00:00,  2.72s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 53.92it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:09<00:00,  2.28s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 24.88it/s]

Inference Embeddings: 100%|██████████| 10/10 [01:24<00:00,  8.42s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 83.14it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.76s/it]

Inference Embeddings: 100%|██████████| 3/3 [01:19<00:00, 26.61s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 92.95it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:06<00:00,  1.56s/it]

pre tokenize: 100%|██████████| 12/12 [00:00<00:00, 43.48it/s]

Inference Embeddings: 100%|██████████| 36/36 [01:35<00:00,  2.67s/it]

pre tokenize

30089
7916
periti_2025
      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  word  immaturity , and want of experience -- besides...   
4  part  immaturity , and want of experience -- besides...   

                                                   2  
0                    (Figurative) A leader or chief.  
1  (Figurative) The most important part or piece ...  
2  The smallest discrete unit of written language...  
3    (Obsolete) Something spoken; speech, discourse.  
4  (Obsolete) A third; a third part; a share, a p...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 73.38it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:07<00:00,  2.58s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 73.56it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:10<00:00,  2.57s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 42.24it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:15<00:00,  3.90s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 113.34it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:06<00:00,  2.03s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 114.87it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:06<00:00,  3.09s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 69.53it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:07<00:00,  3.53s/it]

pre tokenize: 100%|██████████| 8/8 [00:00<00:00, 82.55it/s]

Inference Embeddings: 100%|██████████| 8/8 [00:12<00:00,  1.55s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 60.57it/s]

Inference Embeddings: 100%

22924
6577


In [ ]:
method = "greedy"
new_dir = os.path.join(eng_path, f"{method}-clustered-sep/")
os.makedirs(new_dir, exist_ok=True)
for n in (1, 2):
    print(n)
    method_result = cluster_sep(method, eng_path, n)
    method_result["2"] = method_result["labels"]
    method_result.drop("labels", inplace=True, axis=1)
    method_result.to_csv(f"{new_dir}english-corpus{n}.tsv.gz", sep='\t', index=False, compression="gzip")
    method_result.head()

1


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 109.92it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 109.89it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.70s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 126.94it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 84.74it/s]

Inference Embeddings: 100%|██████████| 5/5 [00:06<00:00,  1.35s/it]

pre tokenize: 100%|██████████| 11/11 [00:00<00:00, 84.34it/s]

Inference Embeddings: 100%|██████████| 11/11 [00:13<00:00,  1.19s/it]

pre tokenize: 100%|██████████| 7/7 [00:00<00:00, 86.34it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:08<00:00,  1.25s/it]

pre tokenize: 100%|██████████| 7/7 [00:00<00:00, 68.39it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:06<00:00,  1.08it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 91.05it/s]

Inference Embeddings:

16420
3718
2


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 111.88it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:04<00:00,  1.47s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 53.04it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:05<00:00,  1.73s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 118.55it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:04<00:00,  1.65s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 112.42it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:04<00:00,  2.14s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 145.66it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.14s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 88.88it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:04<00:00,  1.49s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 45.11it/s]

Inference Embeddings: 100%|██████████| 6/6 [00:07<00:00,  1.26s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 107.50it/s]

Inference Embeddings: 1

18672
4640


In [ ]:
for method in ("greedy", "periti_2024", "periti_2025"):
    print(method)
    new_dir = os.path.join(eng_path, f"{method}-clustered-freq/")
    os.makedirs(new_dir, exist_ok=True)

    data1, data2 = cluster(method, eng_path)
    data1["2"] = data1["labels"]
    data1 = data1[["0", "1", "2"]]
    data1.to_csv(f"{new_dir}english-corpus1.tsv.gz", sep='\t', index=False, compression="gzip")

    data2["2"] = data2["labels"]
    data2 = data2[["0", "1", "2"]]
    data2.to_csv(f"{new_dir}english-corpus2.tsv.gz", sep='\t', index=False, compression="gzip")


greedy
      0                                                  1  \
0  part  Most of the asp * ing men of that part of Alab...   
1  face  The faces are umajured , except that the color...   
2   tip  The moon was in its western field , white and ...   
3  tree  The well known fact that we may have a hundred...   
4  face  Rising upon the instant in his stirrups , the ...   

                                                   2  
0              A portion ; some portion ; a section.  
1                   The front surface of a painting.  
2                  To cover the top of ; to overlay.  
3  A plant that bears a given fruit or seed ; spe...  
4  The front of the skull, especially the front o...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 108.94it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:04<00:00,  1.16s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 98.42it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.34s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 54.91it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.37s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 132.63it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:04<00:00,  1.45s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 105.98it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 94.74it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:05<00:00,  1.29s/it]

pre tokenize: 100%|██████████| 9/9 [00:00<00:00, 44.03it/s]

Inference Embeddings: 100%|██████████| 9/9 [00:10<00:00,  1.13s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 51.46it/s]

Inference Embeddings: 100

33065
7119
periti_2024
      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  part  immaturity , and want of experience -- besides...   
4  word  immaturity , and want of experience -- besides...   

                                                   2  
0            A superior or governing body or member.  
1  The top of the head of a person, or the upper ...  
2  A unit of language that native speakers can id...  
3                             A fraction of a whole.  
4  A single distinct conceptual unit of language,...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 70.68it/s]

Inference Embeddings: 100%|██████████| 12/12 [01:23<00:00,  6.93s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 43.05it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:10<00:00,  2.72s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 58.19it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:09<00:00,  2.27s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 22.90it/s]

Inference Embeddings: 100%|██████████| 10/10 [01:24<00:00,  8.41s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 81.63it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.77s/it]

Inference Embeddings: 100%|██████████| 3/3 [01:19<00:00, 26.63s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 53.69it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:06<00:00,  1.55s/it]

pre tokenize: 100%|██████████| 12/12 [00:00<00:00, 39.88it/s]

Inference Embeddings: 100%|██████████| 36/36 [01:36<00:00,  2.67s/it]

pre tokenize

30089
7916
periti_2025
      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  word  immaturity , and want of experience -- besides...   
4  part  immaturity , and want of experience -- besides...   

                                                   2  
0                    (Figurative) A leader or chief.  
1  (Figurative) The most important part or piece ...  
2  The smallest discrete unit of written language...  
3    (Obsolete) Something spoken; speech, discourse.  
4  (Obsolete) A third; a third part; a share, a p...  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 89.39it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:07<00:00,  2.58s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 44.74it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:10<00:00,  2.57s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 48.53it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:15<00:00,  3.90s/it]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 65.86it/s]

Inference Embeddings: 100%|██████████| 3/3 [00:06<00:00,  2.03s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 109.95it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:06<00:00,  3.09s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 67.23it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:07<00:00,  3.52s/it]

pre tokenize: 100%|██████████| 8/8 [00:00<00:00, 86.01it/s]

Inference Embeddings: 100%|██████████| 8/8 [00:12<00:00,  1.55s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 58.39it/s]

Inference Embeddings: 100%|

22924
6577


In [ ]:
method = "dpo"
new_dir = os.path.join(eng_path, f"{method}-clustered-freq/")
os.makedirs(new_dir, exist_ok=True)

data1, data2 = cluster(method, eng_path)
data1["2"] = data1["labels"]
data1 = data1[["0", "1", "2"]]
data1.to_csv(f"{new_dir}english-corpus1.tsv.gz", sep='\t', index=False, compression="gzip")

data2["2"] = data2["labels"]
data2 = data2[["0", "1", "2"]]
data2.to_csv(f"{new_dir}english-corpus2.tsv.gz", sep='\t', index=False, compression="gzip")


      0                                                  1  \
0  head  seen a complex and exquisitely balanced polity...   
1  head  inculcated doctrines incompatible with the pla...   
2  word  , during all this period , once entering those...   
3  word  immaturity , and want of experience -- besides...   
4  part  immaturity , and want of experience -- besides...   

                                                   2  
0            The topmost, foremost, or leading part.  
1  (Figurative) The most important part or one of...  
2  The smallest discrete unit of written language...  
3                     (Obsolete) A proverb or motto.  
4                  A fragment of something; a piece.  


  0%|          | 0/37 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 78.72it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.94s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 87.38it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:09<00:00,  2.33s/it]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 105.25it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:09<00:00,  2.37s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 111.68it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.55s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 52.17it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:05<00:00,  2.72s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 100.59it/s]

Inference Embeddings: 100%|██████████| 6/6 [00:09<00:00,  1.56s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 62.05it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:07<00:00,  3.98s/it]

pre tokenize: 100%|██████████| 9/9 [00:00<00:00, 94.40it/s]

Inference Embeddings: 100

18869
5777
